# Example of Photometry

For each MCMC sample, the corresponding profile parameters can be input into Lenstronomy's `image_linear_solve` function to determine the optimal fluxes of all light components (e.g., lens light, source light, and quasar images if applicable). To obtain morphological properties for multi-component lens light models, an instance of Lenstronomy's `LightProfileAnalysis` class is instantiated.

# Setting up the Arguments

The `Photometry` class requires the following inputs:

1) An instance of the `Output` class (*note: the data for the specific model must also be loaded via `output.load_output`*)
2) A dictionary mapping the data filters and desired model components to utilize in the linear inversion. For the lensed_quasar example, we have 1 lens light profile, 1 source light profile, and 4 point sources to optimize. `Photometry` already handles point sources, so we only need to specify the indices of the lens light and source light components that we want to keep in the optimization, per filter, as seen below. If we also had, for example, a UNIFORM lens light profile, then we would exclude that from the lens light flux measurement via `exclude_lens_light_indices: [1]`. But note, the index would also need to be included in `lens_light_indices` so that the Linear Solver reconstructs the proper model configuration: `lens_light_indices: [0, 1]`.
3) The model ID of the run to optimize.
4) The walker ratio used in the MCMC process.
5) *optional:* The number of burn-in iterations to discard from the MCMC.
6) *optional* The radius in arcseconds of a circular aperture in which the inversion is to be evaluated.
7) *optional* The length in arcseconds of a circular aperture in which the inversion is to be evaluated.
8) *optional* **do_morphology**: Boolean that dictates whether or not morphological properties of multi-component lens light profiles are evaluated.

# Imports

In [1]:
from dolphin.analysis.output import Output
from dolphin.analysis.photometry import Photometry
import numpy as np

# create Output instance

In [2]:
name = "lensed_quasar"
ID = "example"

output = Output("../io_directory_example/")
output.load_output(f"{name}", f"{ID}")

Loaded output for lensed_quasar with model ID example.
dolphin version used: 1.4.0
lenstronomy version used: 1.13.6


{'settings': {'lens_name': 'lensed_quasar',
  'band': ['F814W'],
  'model': {'lens': ['EPL', 'SHEAR_GAMMA_PSI'],
   'lens_light': ['SERSIC_ELLIPSE'],
   'source_light': ['SERSIC_ELLIPSE'],
   'point_source': ['LENSED_POSITION']},
  'lens_options': {'centroid_init': [-0.2, 0.04]},
  'lens_light_options': {'fix': {'0': {'n_sersic': 4.0}}},
  'source_light_options': {'n_max': [4]},
  'point_source_options': {'ra_init': [-0.54, -0.69, 0.19, 0.55],
   'dec_init': [-0.48, 0.54, 0.68, -0.16],
   'bound': 0.1},
  'fitting': {'pso': True,
   'pso_settings': {'num_particle': 20, 'num_iteration': 50},
   'psf_iteration': True,
   'psf_iteration_settings': {'stacking_method': 'median',
    'num_iter': 20,
    'psf_iter_factor': 0.5,
    'keep_psf_variance_map': True,
    'psf_symmetry': 4,
    'block_center_neighbour': 0.2},
   'sampling': True,
   'sampler': 'emcee',
   'sampler_settings': {'n_burn': 0,
    'n_run': 100,
    'walkerRatio': 2,
    'threadCount': 1,
    'init_samples': None}},
  'n

# create band_config dictionary

In [3]:
band_config = {
    "F814W": {
        "lens_light_indices": [0],
        "source_indices": [0],
        "exclude_lens_light_indices": [],
    }
}

# If we had multiple filters, it would look like so:

# band_config = {
#    "F814W": {
#        "lens_light_indices": [0],
#        "source_indices": [0],
#        "exclude_lens_light_indices": [],
#    },
#    "F475X": {
#        "lens_light_indices": [1],
#        "source_indices": [1],
#        "exclude_lens_light_indices": [],
#    }
# }

# create Photometry instance

For this example, we will only use the last 25 steps of the MCMC. We will also not use a circular/square aperture, and instead evaluate over the full image grid. Even though this system uses one lens light model, we will still solve for the light morphology for the purposes of this example.

In [4]:
phot = Photometry(
    output,
    band_config=band_config,
    model_id=ID,
    walker_ratio=2,
    burn_in=-25,
    aperture_radius=None,
    aperture_length=None,
    do_morphology=True,
)

Loaded output for lensed_quasar with model ID example.
dolphin version used: 1.4.0
lenstronomy version used: 1.13.6


# perform linear inversion, convert to magnitudes, and save outputs

By default, Lenstronomy's linear inversion returns fluxes in units in which the data is taken. As such, one can convert these to magnitudes by hand or use the helper functions implemented in Dolphin. Currently, the following instruments are supported (alongside the required calibration parameters from their FITS headers):

1) JWST:
- `pixar_sr`: JWST average pixel area in units of steradians

2) HST:
- `photflam`: mean flux density (in erg cm–2 sec–1 Å–1) that produces 1 count per second in the HST observing mode
- `photplam`: HST filter pivot wavelength

The fluxes are converted to AB magnitudes as follows:

# 1) For JWST data

For JWST images, the pixel flux is converted into Janskys using the average pixel area:

$$
F_{\rm Jy}
=
F
\times
\mathrm{PIXAR\_SR}
\times
10^6,
$$

The AB magnitude is then computed directly from the standard definition:

$$
m_{\rm AB}
=
-2.5\log_{10}
\left(
\frac{F_{\rm Jy}}{3631}
\right),
$$

where 3631 Jy is the AB magnitude zero point flux density.

# 2) For HST data

The pixel flux returned by the linear inversion is first converted into physical flux density units:

$$
F_\lambda = F \times \mathrm{PHOTFLAM},
$$

The corresponding ST magnitude is then computed as

$$
m_{\rm ST}
=
-2.5\log_{10}(F_\lambda)
- 21.1,
$$


Finally, the ST magnitude is converted into an AB magnitude:

$$
m_{\rm AB}
=
m_{\rm ST}
-
5\log_{10}(\mathrm{PHOTPLAM})
+
2.5\log_{10}(c)
-
27.5,
$$

To convert the output fluxes to magnitudes, one must pass a dictionary that maps the data filters to their respective calibration parameters. An example is given below.

In [5]:
flux_chain, morph = phot.do_linear_inversion()

magnitude_config = {"F814W": {"photflam": 1.52122335e-19, "photplam": 8034.189}}

mag_chain = phot.calculate_ab_magnitude(
    flux_chain=flux_chain, magnitude_config=magnitude_config
)

phot.save_to_hdf5(flux_chain, mag_chain, morph)

Loaded output for lensed_quasar with model ID example.
dolphin version used: 1.4.0
lenstronomy version used: 1.13.6
Instrument identified as: HST


# Load the outputs and analyze some statistics

In [6]:
flux_chain = phot.load_flux_chain()
mag_chain = phot.load_magnitude_chain()
morph_chain = phot.load_morphology_chain()

# Image 1, the zeroth index:
print(f"Image 1 Average Magnitude: {np.mean(mag_chain[:, 0])}")

# Lens, the fourth index:
print(f"Lens Average Magnitude: {np.mean(mag_chain[:, 4])}")

# Extended Source, the fifth index:
print(f"Extended Source Average Magnitude: {np.mean(mag_chain[:, 5])}")

# Instrinsic Source, the sixth index:
print(f"Intrinsic Source Average Magnitude: {np.mean(mag_chain[:, 6])}")

Image 1 Average Magnitude: 20.397287764777772
Lens Average Magnitude: 21.57967985869685
Extended Source Average Magnitude: 21.391098277803636
Intrinsic Source Average Magnitude: 25.04769601095083
